<a href="https://colab.research.google.com/github/fatimaezzahrabakas61-web/meteo_project/blob/main/datasetbilanhyd_pipclean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd


# 1. Charger les données
def load_data(path):
    return pd.read_excel(path)

# 2. Convertir en numérique (avec gestion des virgules)

def convert_columns_to_numeric(df, cols):
    for col in cols:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(",", "."),
            errors="coerce"
        ).fillna(0)
    return df

# 3. Supprimer colonnes inutiles
def drop_columns(df):
    cols_to_drop = ["Resti_TOTAL"]
    return df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")

# 4. Ajouter la colonne transfert_eau
def add_transfert_eau(df):
    def calc_transfert(val):
        val = str(val)
        if val == "2022-2023":
            return 20
        elif val == "2021-2022":
            return 40
        return 0

    if "C_A" in df.columns:
        df["transfert_eau"] = df["C_A"].apply(calc_transfert)
    else:
        df["transfert_eau"] = 0

    return df

# 5. Calculer Resti_total
def compute_resti_total(df):
    resti_cols = [
        "Resti_Irrig_Turbinage",
        "Resti_Turbinages_Exclusifs",
        "Resti_Déver_Evacuation",
        "Resti_ONEP",
        "Resti_Massen"
    ]

    # ✅ assurer que toutes les colonnes existent
    for col in resti_cols:
        if col not in df.columns:
            df[col] = 0

    df["Resti_total"] = df[resti_cols].sum(axis=1)

    return df

 # 7. Sauvegarder
def save_data(df, path):
    df.to_excel(path, index=False)








In [2]:
# 8. Pipeline complet
def pipeline(input_path, output_path):
    df = load_data(input_path)
    cols = [
    "Resti_Irrig_Turbinage",
    "Resti_Turbinages_Exclusifs",
    "Resti_Déver_Evacuation",
    "Resti_ONEP",
    "Resti_Massen",
    "Stock_au_de_C.A_1erSep",
    "Apports",
    "Evaporation"
]

    df = convert_columns_to_numeric(df, cols)
    df = drop_columns(df)
    df = add_transfert_eau(df)
    df = compute_resti_total(df)
    save_data(df, output_path)


In [3]:
# ▶️ Exécution
if __name__ == "__main__":
   input_file = "copy-hydraulique_touse.xlsx"
   output_file = "dataset_bilanhyd_clean.xlsx"

   pipeline(input_file, output_file)